# Phase 1 — SimCLR reference encoder (Kaggle)

In [ ]:
!nvidia-smi

## 1. Clone the repo

Set `PROBE_REPO_URL` to your public Git URL or `PROBE_REPO_DIR` to a local checkout.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = os.environ.get("PROBE_REPO_DIR", "/kaggle/working/probe-capacity-invariance")
if not (Path(REPO_DIR) / "pyproject.toml").is_file():
    repo_url = os.environ.get("PROBE_REPO_URL")
    if not repo_url:
        raise RuntimeError("Set PROBE_REPO_URL or PROBE_REPO_DIR before running this notebook.")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)


## 2. Install dependencies

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Settings > Accelerator > GPU")

## 3. Download Shapes3D + build the image cache

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 4. Calibrate runtime (optional, one epoch)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml \
    --set run.seed=0 run.epochs=1 run.num_workers=2

## 4b. Resume from a downloaded checkpoint (optional)

In [ ]:
import shutil
from pathlib import Path



RUN_IDS = ["standard_strong_seed0"]

found = sorted(Path("/kaggle/input").rglob("last_ckpt.pt"))
print(f"found {len(found)} checkpoint file(s):")
for f in found:
    print(" ", f)

if len(found) != len(RUN_IDS):
    print(
        f"\nWARNING: found {len(found)} checkpoint(s) but RUN_IDS has "
        f"{len(RUN_IDS)} entries — fix RUN_IDS to match, then rerun this cell."
    )
else:
    for src, run_id in zip(found, RUN_IDS):
        dest_dir = Path("results/encoders") / run_id
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / "last_ckpt.pt"
        shutil.copy(src, dest)
        print(f"restored {src} -> {dest}")

## 4c. (Fast path) Reuse encoders you already trained

In [ ]:
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")



RUN_IDS = ["standard_strong_seed0", "standard_strong_seed1"]

for run_id in RUN_IDS:
    hits = [p for p in Path("/kaggle/input").rglob("*.pt")
            if run_id in p.as_posix() and (p.name == "backbone.pt" or run_id in p.name)]
    if not hits:
        print(f"— {run_id}: no backbone.pt found under /kaggle/input — train it below")
        continue
    dst = REPO / "results" / "encoders" / run_id / "backbone.pt"
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(hits[0], dst)
    print(f"✓ {run_id}: restored from {hits[0]} ({dst.stat().st_size // 1024 // 1024} MB)")

print("\nRestored backbones let you SKIP their training sections. "
      "Then run section 7 (supervised) if not restored, and section 8 (the gate).")

## 5. Train the reference encoder — seed 0

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml \
    --set run.seed=0 run.num_workers=2

## 6. Train the reference encoder — seed 1 (seed-to-seed reproducibility check)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml \
    --set run.seed=1 run.num_workers=2

## 7. Train the supervised-reference encoder (recoverability ceiling)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.supervised \
    --config configs/run/supervised.yaml \
    --set run.seed=0 run.num_workers=2

## 8. Run the encoder-quality gate

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.eval.quality_gate \
    --config configs/run/reference_cuda.yaml \
    --simclr results/encoders/standard_strong_seed0/backbone.pt \
             results/encoders/standard_strong_seed1/backbone.pt \
    --supervised results/encoders/supervised_seed0/backbone.pt \
    --random-seed 0 \
    --out results/quality_gate/reference.json

## 9. Print the gate report

In [ ]:
import json
print(json.dumps(json.load(open("results/quality_gate/reference.json")), indent=2))

## 10. Phase 2 — validate the probe instrument on controls

In [ ]:
from pathlib import Path

assert Path('pyproject.toml').is_file(), 'Run the repository setup cell first.'

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.validate \
    --config configs/probe/ladder.yaml \
    --simclr results/encoders/standard_strong_seed0/backbone.pt \
             results/encoders/standard_strong_seed1/backbone.pt \
    --random-seed 0 1 2 \
    --subsample 8000 --epochs 60 --num-workers 2 \
    --out results/probes/phase2_validation.json

## 11. Print the Phase-2 validation report

In [ ]:
import json
r = json.load(open("results/probes/phase2_validation.json"))
print("VERDICT:", "PASS" if r["checks"]["passed"] else "FAIL")
print(json.dumps(r["checks"], indent=2))
print("\nepsilon source:", r["report"]["epsilon_source"], "| flips_primary:", r["report"]["flips_primary"])
for fac, rows in r["report"]["table"].items():
    print(fac, [(c["rung"], round(c["G"], 3), round(c["S"], 3), c["case"]) for c in rows])